In [1]:
import yaml
import json
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
import pprint
import google.generativeai as genai
import json
import re

# 指定 log 檔案
cu_log_file = "/home/aiml/johnson/Scenario/Scenario_6/1_cu_gnb_Num_Threads_PUSCH_log.txt"
# du_log_file = "/home/aiml/johnson/Scenario/Scenario_For_testing/DU/log/du.log"
# ru_log_file = "/home/aiml/johnson/Scenario/Scenario_For_testing/RU/log/RU.log"

# pcap_path = "/home/aiml/johnson/Scenario/Scenario_For_testing/FH/fh.pcap"
debug_yaml_path = "/home/aiml/johnson/thesis_rag/Integration_dataset/debug.yaml"
reference_context_path = "/home/aiml/johnson/Scenario/Scenario_6/reference_config.txt"

current_cu_config_path="/home/aiml/johnson/Scenario/Scenario_6/1_cu_gnb_Num_Threads_PUSCH.conf"
# current_du_config_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/du.conf"

current_cu_config_json_path="/home/aiml/johnson/Scenario/Scenario_6/1_cu_gnb_Num_Threads_PUSCH.conf.segments.json"
# current_du_config_json_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/du_conf.json"


rag_after_cu_conf_path="/home/aiml/johnson/Scenario/Scenario_6/CU/conf/Scenario_6_cu_modification_1.conf"
# rag_after_du_conf_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/Scenario_For_testing_du_modification_1.conf"

rag_after_cu_json_path="/home/aiml/johnson/Scenario/Scenario_6/CU/conf/Scenario_6_cu_modification_1.conf.segments.json"
# rag_after_du_json_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/Scenario_For_testing_du_modification_1.conf.segments.json"

In [2]:
def parse_llm_response(response_text):
    # 取出 JSON 區塊（包在 ```json ... ``` 中）
    match = re.search(r"```json\s*(\[\s*{.*?}\s*\])\s*```", response_text, re.DOTALL)
    if match:
        try:
            suggestions = json.loads(match.group(1))
            return suggestions
        except json.JSONDecodeError as e:
            print("❌ JSON decode error:", e)
            return []
    else:
        print("⚠️ No JSON block found in LLM response.")
        return []

In [3]:
with open( debug_yaml_path , "r") as f:
    debug_data = yaml.safe_load(f)

# The embedding format for each entry (based on symptom and log as the primary content)
embedding_docs = []
for item in debug_data:
    content = f"Stage: {item['stage']}\nSymptom: {item['symptom']}\nLog: {item['log_snippet']}\n"

    if "notes" in item and item["notes"]:
        content += f"Notes: {item['notes']}\n"

    related_config_str = ", ".join(item["related_config"])  # ✅ Convert list to comma-separated string
    metadata = {
        "stage": item["stage"],
        "related_config": related_config_str
    }
    embedding_docs.append({"content": content, "metadata": metadata})

# pprint.pprint(embedding_docs) #for checking

In [4]:
# 你也可以改用 Gemini 或 OpenAI embedding
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 將文本嵌入向量並存入 Chroma 資料庫
texts = [d["content"] for d in embedding_docs]
metadatas = [d["metadata"] for d in embedding_docs]

vectordb = Chroma.from_texts(texts, embedding=embedding, metadatas=metadatas, persist_directory="./error_db")
vectordb.persist()

print("✅ Debug embedding 建立完成並已儲存")



# 檢查嵌入總筆數
print("📦 總筆數：", vectordb._collection.count())
# 顯示前幾筆嵌入資料內容（包括原始文本與 metadata）
peek_data = vectordb._collection.get(limit=1)

for i in range(len(peek_data["documents"])):
    print(f"\n--- Entry {i+1} ---")
    print("Document ID:", peek_data["ids"][i])
    print("Document Text:", peek_data["documents"][i])
    print("Metadata:", peek_data["metadatas"][i])

/tmp/ipykernel_1136389/1403688346.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
2025-05-26 09:11:16.744221: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 09:11:16.763128: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already

✅ Debug embedding 建立完成並已儲存
📦 總筆數： 7

--- Entry 1 ---
Document ID: 319f59b5-0f07-4fe2-9e2f-44e56aba0062
Document Text: Stage: cu_init
Symptom: Invalid or non-integer value for Num_Threads_PUSCH caused configuration parsing failure
Log: Num_Threads_PUSCH.conf - line
Notes: The value assigned to Num_Threads_PUSCH must be a valid integer. Recommended value: 8

Metadata: {'related_config': 'Num_Threads_PUSCH', 'stage': 'cu_init'}


/tmp/ipykernel_1136389/1403688346.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


## Check CU log

In [5]:
import yaml
import re
from pathlib import Path

def clean_text(s):
    """去除ANSI控制字元 + 移除引號 + 去除多餘空格"""
    ansi_escape = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
    s = ansi_escape.sub('', s)
    s = s.replace("'", "").replace('"', "")
    s = s.strip()
    return s

if not Path(cu_log_file).exists():
    raise FileNotFoundError(f"Log file not found: {cu_log_file}")
if not Path(debug_yaml_path).exists():
    raise FileNotFoundError(f"Debug YAML not found: {debug_yaml_path}")

# 讀取 debug.yaml
# Read debug.yaml
with open(debug_yaml_path, 'r', encoding='utf-8') as f:
    debug_data = yaml.safe_load(f)

# 整理出 (log_snippet, stage) 對應
# Organize (log_snippet, stage) pairs
target_entries = []
for item in debug_data:
    if 'log_snippet' in item:
        snippet = item['log_snippet']
        stage = item.get('stage', 'unknown')
        if ";" in snippet:
            parts = [s.strip() for s in snippet.split(";")]
            for p in parts:
                target_entries.append((p, stage))
        else:
            target_entries.append((snippet.strip(), stage))

# 搜索 log
# search log
found_results = []
with open(cu_log_file, 'r', encoding='utf-8', errors='ignore') as f:
    for raw_line in f:
        line = clean_text(raw_line)
        for snippet, stage in target_entries:
            snippet_cleaned = clean_text(snippet)
            # 只要關鍵字部分包含就算符合
            # As long as the keyword is partially matched, it is considered a match.
            if snippet_cleaned in line:
                found_results.append((stage, snippet))
                
# 輸出結果
# output results
if found_results:
    for stage, snippet in found_results:
        print(f"✅ Found matching log for stage [{stage}]: {snippet}")
        query = snippet
else:
    print("❌ No matching logs found.")


✅ Found matching log for stage [cu_init]: Num_Threads_PUSCH.conf - line


In [ ]:
results = vectordb.similarity_search(query, k=2)

for r in results:
    print("- Matched:", r.page_content)
    print("- Related config:", r.metadata["related_config"])
    print("-----------------------------------------------")

matched_case = results[0]
matched_symptom = matched_case.page_content
matched_related_config = matched_case.metadata.get("related_config", "")



- Matched: Stage: cu_init
Symptom: Invalid or non-integer value for Num_Threads_PUSCH caused configuration parsing failure
Log: Num_Threads_PUSCH.conf - line
Notes: The value assigned to Num_Threads_PUSCH must be a valid integer. Recommended value: 8

- Related config: Num_Threads_PUSCH
-----------------------------------------------
- Matched: Stage: du_initialization
Symptom: DU failed to correctly parse the gNB configuration, resulting in a 'no active gNB found/mismatch of gNBs' error and abnormal process termination.
Log: Assertion (strcmp(GNBSParams[1].strlistptr[0], *GNBParamList.paramarray[0][2].strptr) == 0) failed! ... no active gNB found/mismatch of gNBs: gNB-Test-DU vs gNB-Eurecom-DU
Notes: 1. Check whether the Active_gNBs entry in the DU configuration matches the actual gNB_name definitions.
2. This error is commonly caused by mismatches in multi-gNB configurations or unsynchronized manual naming.


- Related config: Active_gNBs, gNB_name
-----------------------------------

In [7]:
with open(current_cu_config_json_path, "r") as f:
    config_cu_segments_context = json.load(f)
# with open(current_du_config_json_path, "r") as f:
#     config_du_segments_context = json.load(f)
# with open(current_ru_config_json_path, "r") as f:
#     config_ru_segments_context = json.load(f)


with open(reference_context_path, "r") as f:
    reference_context = f.read()

# RAG prompt_template
rag_prompt_template = f"""
You are a 5G network expert. Your job is to revise configuration files based on observed network issues and debug knowledge.

Issue Description:
"{query}"

Matching debug knowledge:
{matched_case.page_content}
Relevant parameters: {matched_case.metadata["related_config"]}

Reference Device Address Table (external reference file):
{reference_context}

Current CU configuration block:
{config_cu_segments_context}


Please revise the configuration using correct addresses from the reference. Output only the revised config section.

Return a list of JSON objects with the following structure:
[
  {{
    "label": "parameter_name",
    "content": "parameter_name = (...);",
    "reference_reason": "Short explanation matching the value to the reference device table (e.g., correct MAC, matches expected setting).",
    "model_reason": "Additional expert analysis in 1-2 sentences explaining why this change is necessary, beneficial, or resolves a network issue."
    "target": "CU" or "DU" or "RU" or "FH"
  }},
  ...
]

- Only include parameters listed in 'Relevant parameters'.
- Do not include any explanation outside of the JSON structure.
- Keep "reference_reason" based on the reference table.
- Derive "model_reason" from your own technical reasoning.
- Set the "target" field based on the location of the parameter: "CU" for CU config, "DU" for DU config, "RU" for RU config, and "FH" for FH config.
- If multiple configuration problems exist at the same time, return multiple JSON objects — one for each necessary change.

"""

none_rag_prompt_template = f"""
You are a 5G network expert. Your job is to revise configuration files based on observed network issues.

Issue Description:
"{query}"

Current configuration block:
{config_cu_segments_context}

Please revise the configuration to resolve the described issue based on your technical expertise.

Return a list of JSON objects with the following structure:
[
  {{
    "label": "parameter_name",
    "content": "parameter_name = (...);",
    "model_reason": "Technical explanation in 1-2 sentences explaining why this change is necessary, beneficial, or resolves the network issue."
  }},
  ...
]

- Only revise parameters that are necessary to resolve the issue.
- If no changes are needed, return an empty list: []
- Strictly output only valid JSON without any additional text or explanation.
"""



## Checkpoint


In [ ]:
import os
reason_output_dir = "Reason"
os.makedirs(reason_output_dir, exist_ok=True)


def save_json(filename, data):
    with open(os.path.join(reason_output_dir, filename), "w", encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)




genai.configure(api_key="xxxx") #set your API key here
model = genai.GenerativeModel("gemini-2.0-flash")


none_rag_response = model.generate_content(none_rag_prompt_template)                                # Gemini API
# LLM Suggested Revisions
print("LLM Suggested Revisions：\n")
print(none_rag_response.text)

none_rag_llm_suggestions = parse_llm_response(none_rag_response.text)
print("========= Suggestions =========")
print(none_rag_llm_suggestions)



rag_response = model.generate_content(rag_prompt_template)                                # Gemini API
# LLM Suggested Revisions
print("LLM Suggested Revisions：\n")
print(rag_response.text)

rag_llm_suggestions = parse_llm_response(rag_response.text)
print("========= Suggestions =========")
print(rag_llm_suggestions)



save_json(f"{matched_related_config}_rag.json", rag_llm_suggestions)
save_json(f"{matched_related_config}_none_rag.json", none_rag_llm_suggestions)


LLM Suggested Revisions：

```json
[
  {
    "label": "Num_Threads_PUSCH",
    "content": "Num_Threads_PUSCH = 4;",
    "model_reason": "The value for Num_Threads_PUSCH is currently invalid ('asdasfsad'). It should be an integer representing the number of threads to use for PUSCH processing. Setting it to 4 provides a reasonable balance between resource utilization and processing capacity, suitable for many gNB deployments."
  }
]
```
========= Suggestions =========
[{'label': 'Num_Threads_PUSCH', 'content': 'Num_Threads_PUSCH = 4;', 'model_reason': "The value for Num_Threads_PUSCH is currently invalid ('asdasfsad'). It should be an integer representing the number of threads to use for PUSCH processing. Setting it to 4 provides a reasonable balance between resource utilization and processing capacity, suitable for many gNB deployments."}]
LLM Suggested Revisions：

```json
[
  {
    "label": "Num_Threads_PUSCH",
    "content": "Num_Threads_PUSCH = 8;",
    "reference_reason": "Recommende

In [19]:
import re
import difflib
import json
import os


def safe_print(text):
    try:
        print(text.encode('utf-8', 'replace').decode('utf-8'))
    except Exception:
        print("[Output error suppressed]")


def compare_conf_files(original_path, modified_path, diff_log_path):
    with open(original_path, 'r', encoding='utf-8') as f1, open(modified_path, 'r', encoding='utf-8') as f2:
        original_lines = f1.readlines()
        modified_lines = f2.readlines()

    diff = list(difflib.unified_diff(
        original_lines,
        modified_lines,
        fromfile=original_path,
        tofile=modified_path,
        lineterm=''
    ))

    if diff:
        diff_text = '\n'.join(diff)

        with open(diff_log_path, 'w', encoding='utf-8') as log_file:
            log_file.write(diff_text)

        safe_print("🧾 Differences detected:")
        safe_print(diff_text)
        safe_print(f"\n📄 Diff written to: {diff_log_path}")
    else:
        safe_print("✅ No differences found between the two config files.")


def apply_llm_suggestions(conf_path, output_path, llm_suggestions, config_type):
    with open(conf_path, "r", encoding="utf-8") as f:
        content = f.read()

    modified_labels = []
    change_log = []

    for suggestion in llm_suggestions:
        label = suggestion["label"]
        replacement = suggestion["content"]
        model_reason = suggestion.get("model_reason", "")

        pattern = rf"{label}\s*=\s*.*?;"
        match = re.search(pattern, content, flags=re.DOTALL)

        if match:
            original_line = match.group(0).strip()
            if original_line != replacement.strip():
                content = re.sub(pattern, replacement, content, flags=re.DOTALL)
                modified_labels.append(label)
                change_log.append((label, original_line, replacement.strip(), model_reason))
            else:
                safe_print(f"ℹ️ [{config_type}] {label} already matches suggested value.")
        else:
            safe_print(f"⚠️ [{config_type}] No matching setting found: {label}")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)

    safe_print(f"✅ [{config_type}] Updated file: {output_path}")

    if modified_labels:
        safe_print(f"🛠️ [{config_type}] Modified parameters:")
        for label in modified_labels:
            safe_print(f" - {label}")
    else:
        safe_print(f"📭 [{config_type}] No parameters were modified")

    return content, modified_labels, change_log


def split_suggestions_by_target(llm_suggestions):
    cu_suggestions = []
    du_suggestions = []
    for s in llm_suggestions:
        if s.get("target") == "CU":
            cu_suggestions.append(s)
        elif s.get("target") == "DU":
            du_suggestions.append(s)
    return cu_suggestions, du_suggestions


def save_modified_config(content, output_path, config_type):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)
    safe_print(f"✅ [{config_type}] Updated file: {output_path}")


def save_sft_data(change_log, output_path, config_type, suggestions, current_path):
    reason_map = {s["label"]: s.get("reference_reason", "") for s in suggestions}
    sft_data = []
    for label, before, after, model_reason in change_log:
        sft_data.append({
            "label": label,
            "before": before,
            "after": after,
            "model_reason": model_reason,
            "reference_reason": reason_map.get(label, ""),
            "config_type": config_type,
            "source_file": os.path.basename(current_path)
        })

    json_path = os.path.splitext(output_path)[0] + f"_{config_type}_sft.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(sft_data, f, indent=2, ensure_ascii=False)
    safe_print(f"\n📁 [{config_type}] SFT data saved to: {json_path}")


def print_and_save_change_summary(change_log, config_type, output_path):
    if not change_log:
        safe_print(f"\n👭 [{config_type}] No parameters were modified.")
        return

    safe_print(f"\n📋 [{config_type}] Change Summary:")
    summary = []
    for label, before, after, reason in change_log:
        safe_print(f"\n🔧 {label}")
        safe_print(f"Current setting : {before}")
        safe_print(f"Proposed change : {after}")
        safe_print(f"Model reason     : {reason}")
        summary.append({
            "label": label,
            "before": before,
            "after": after,
            "model_reason": reason
        })

    json_path = os.path.splitext(output_path)[0] + f"_{config_type}_summary.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    safe_print(f"\n📁 [{config_type}] Change summary saved to: {json_path}")


def process_config_type(config_type, suggestions, current_path, modified_path, original_path, diff_log_path):
    if not suggestions:
        safe_print(f"📄 No LLM suggestions for {config_type}.")
        return

    content, modified_labels, change_log = apply_llm_suggestions(
        conf_path=current_path,
        output_path=modified_path,
        llm_suggestions=suggestions,
        config_type=config_type
    )

    save_modified_config(content, modified_path, config_type)
    compare_conf_files(original_path, modified_path, diff_log_path)
    print_and_save_change_summary(change_log, config_type, modified_path)
    save_sft_data(change_log, modified_path, config_type, suggestions, current_path)




In [20]:
cu_suggestions, du_suggestions = split_suggestions_by_target(llm_suggestions)
safe_print("CU Suggestions:")
safe_print(cu_suggestions)
safe_print("DU Suggestions:")
safe_print(du_suggestions)

process_config_type(
    config_type="CU",
    suggestions=cu_suggestions,
    current_path=current_cu_config_path,
    modified_path=rag_after_cu_conf_path,
    original_path="/home/aiml/johnson/Scenario/Scenario_6/1_cu_gnb_Num_Threads_PUSCH.conf",
    diff_log_path="/home/aiml/johnson/Scenario/Scenario_6/cu_diff_log.txt"
)

# process_config_type(
#     config_type="DU",
#     suggestions=du_suggestions,
#     current_path=current_du_config_path,
#     modified_path=rag_after_du_conf_path,
#     original_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/du.conf",
#     diff_log_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/du_diff_log.txt"
# )

# process_config_type(
#     config_type="RU",
#     suggestions=du_suggestions,
#     current_path=current_ru_config_path,
#     modified_path=rag_after_ru_conf_path,
#     original_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/du.conf",
#     diff_log_path="/home/aiml/johnson/Scenario/Scenario_For_testing/DU/conf/du_diff_log.txt"
# )

CU Suggestions:
[Output error suppressed]
DU Suggestions:
[Output error suppressed]
✅ [CU] Updated file: /home/aiml/johnson/Scenario/Scenario_For_testing/CU/conf/Scenario_For_testing_cu_modification_1.conf
🛠️ [CU] Modified parameters:
 - Num_Threads_PUSCH
✅ [CU] Updated file: /home/aiml/johnson/Scenario/Scenario_For_testing/CU/conf/Scenario_For_testing_cu_modification_1.conf
🧾 Differences detected:
--- /home/aiml/johnson/Scenario/Scenario_6/1_cu_gnb_Num_Threads_PUSCH.conf
+++ /home/aiml/johnson/Scenario/Scenario_For_testing/CU/conf/Scenario_For_testing_cu_modification_1.conf
@@ -5,7 +5,7 @@
 

 # Asn1_verbosity, choice in: none, info, annoying

 Asn1_verbosity = "none";

-Num_Threads_PUSCH = asdasfsad;

+Num_Threads_PUSCH = 8;

 

 gNBs =

 (


📄 Diff written to: /home/aiml/johnson/Scenario/Scenario_6/cu_diff_log.txt

📋 [CU] Change Summary:

🔧 Num_Threads_PUSCH
Current setting : Num_Threads_PUSCH = asdasfsad;
Proposed change : Num_Threads_PUSCH = 8;
Model reason     : The value 'asdasf